In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
def get_clean_data(file_path):
    df = pd.read_csv(file_path)

    # Explicitly convert relevant columns to numeric, coercing errors to NaN
    # This handles cases where data might be read as strings or contain non-numeric characters
    df['Open'] = pd.to_numeric(df['Open'], errors='coerce')
    df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
    df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')

    # Extract Strike from Ticker (e.g., AARTIIND28NOV24490PE.NFO)
    def parse_strike(ticker):
        match = re.search(r"24(\d+)[CP]E", str(ticker))
        if match:
            return float(match.group(1))
        return np.nan # Return NaN if strike cannot be parsed

    df['Strike'] = df['Ticker'].apply(parse_strike)
    df['Strike'] = pd.to_numeric(df['Strike'], errors='coerce') # Ensure Strike is numeric after parsing

    # Drop NaNs from all critical columns *after* numeric conversion and parsing
    df = df.dropna(subset=['Strike', 'Close', 'Open', 'Volume'])

    # Ensure Strike is not zero to avoid division by zero (which creates inf)
    # Using np.isclose for float comparison for robustness against very small numbers
    df = df[~np.isclose(df['Strike'], 0)]

    # Feature Engineering: Moneyness (S/K) and Time to Expiry (T)
    df['S_K'] = df['Open'] / df['Strike']
    df['T'] = 30 / 365.0  # Constant proxy if specific dates are noisy

    # Re-check for NaNs or Infs introduced during feature engineering (e.g., if Strike was near zero)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['S_K', 'T', 'Volume'])

    y = (df['Close'] / df['Strike']).values # Normalized Target
    y = np.where(np.isfinite(y), y, np.nan) # Coerce any inf in y to NaN

    # Select features for X
    X = df[['S_K', 'T', 'Volume']].values

    # Final check for finite values before scaling
    finite_mask_X = np.isfinite(X).all(axis=1)
    finite_mask_y = np.isfinite(y) # y is already a 1D array from .values

    combined_finite_mask = finite_mask_X & finite_mask_y

    X = X[combined_finite_mask]
    y = y[combined_finite_mask]

    # Check if there is any data left after cleaning
    if X.shape[0] == 0:
        raise ValueError("No finite data remaining after cleaning. Please check your CSV file for problematic values or adjust cleaning steps.")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = get_clean_data('NSE_FNO_DATA_2024-10-21.CSV')

model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=0)

In [ ]:
y_pred = model.predict(X_test).flatten()

mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred)**2)
rmse = np.sqrt(mse)
r2 = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2))

print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.4f}")

7432/7432 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
MAE:  0.000354
RMSE: 0.000662
R2:   0.9985
